In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import gc
import sys
import os
import scipy.stats as stats
import pickle
import lightgbm as lgbm


pd.set_option("display.max_rows", 1000)

sys.path.append("..")
from utils import reduce_mem_usage
from tqdm import tqdm
from loguru import logger
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error

%load_ext nb_black

In [ ]:
def read_data(data_dir):
    sales = pd.read_csv(os.path.join(data_dir, "sales_train_evaluation.csv"))
    calendar = pd.read_csv(os.path.join(data_dir, "calendar.csv"))
    prices = pd.read_csv(os.path.join(data_dir, "sell_prices.csv"))
    submission = pd.read_csv(os.path.join(data_dir, "sample_submission.csv"))

    #     sales = reduce_mem_usage(sales)
    #     calendar = reduce_mem_usage(calendar)
    #     prices = reduce_mem_usage(prices)
    #     submission = reduce_mem_usage(submission)

    return sales, calendar, prices, submission

In [ ]:
sales, calendar, prices, submission = read_data("../data/")

In [ ]:
def create_shifted_feature(
    data: pd.DataFrame, lag: int = 28, feature_col: str = "sales", id_col: str = "id",
) -> pd.DataFrame:
    feat = data.groupby(id_col)[feature_col].shift(-lag).fillna(0)
    return feat


def preprocess_calendar(calendar: pd.DataFrame) -> pd.DataFrame:
    calendar = calendar.copy()
    event_name_map = {mod: i for i, mod in enumerate(calendar["event_name_1"].unique())}
    event_type_map = {mod: i for i, mod in enumerate(calendar["event_type_1"].unique())}

    calendar["event_name"] = calendar["event_name_1"]
    calendar["event_type"] = calendar["event_type_1"]

    #     calendar["event_name"] = calendar["event_name"].map(event_name_map)
    #     calendar["event_type"] = calendar["event_type"].map(event_type_map)
    calendar["d"] = calendar["d"].str.replace("d_", "").astype(int)
    calendar["wday"] -= 1
    calendar["month"] -= 1
    calendar["year"] -= 2011
    calendar["week_number"] = pd.to_datetime(calendar["date"]).dt.week

    calendar = calendar.sort_values("date")
    calendar = calendar.drop(
        [
            "event_name_1",
            "event_name_2",
            "event_type_1",
            "event_type_2",
            #             "date",
            #             "weekday",
        ],
        axis=1,
    )
    return calendar


def preprocess_sales(sales):
    sales = sales.copy()
    series_cols = sales.columns[sales.columns.str.contains("d_")].values
    level_cols = sales.columns[~sales.columns.str.contains("d_")].values
    sales = sales.melt(level_cols, series_cols, var_name="day", value_name="sales")
    sales["day"] = sales["day"].str.replace("d_", "").astype(int)
    return sales


def merge_and_melt(sales, calendar, prices):
    logger.info("preprocessing")
    sales = preprocess_sales(sales)
    calendar = preprocess_calendar(calendar)

    logger.info("merging with calendar, prices")
    # merge with calendar
    sales = sales.merge(
        calendar, left_on="day", right_on="d", validate="many_to_one", how="left"
    )
    sales = sales.merge(
        prices,
        left_on=["item_id", "store_id", "wm_yr_wk"],
        right_on=["item_id", "store_id", "wm_yr_wk"],
        validate="many_to_one",
        how="left",
    )
    return sales


def intervaled_cumsum(a, trigger_val=1, start_val=0, invalid_specifier=-1):
    out = np.ones(a.size, dtype=int)
    idx = np.flatnonzero(a == trigger_val)
    if len(idx) == 0:
        return np.full(a.size, invalid_specifier)
    else:
        out[idx[0]] = -idx[0] + 1
        out[0] = start_val
        out[idx[1:]] = idx[:-1] - idx[1:] + 1
        np.cumsum(out, out=out)
        out[: idx[0]] = invalid_specifier
        return out


def feature_engineering(
    sales: pd.DataFrame, ahead_windows=[3, 7, 14, 28]
) -> pd.DataFrame:

    logger.info("feature engineeering")
    # impute missing data on sell price
    sales["sell_price"] = sales.groupby("item_id")["sell_price"].apply(
        lambda x: x.fillna(x[~x.isnull()].median())
    )

    # make rolling averages/std for items
    windows = [3, 7, 30, 90, 180, 360]
    for window in tqdm(windows, desc="fe: rolling stats"):
        sales.loc[:, f"rolling_sales_mean_{window}"] = (
            sales.groupby(["id"])["sales"]
            .transform(lambda x: x.rolling(window=window).mean())
            .astype(np.float16)
            .fillna(0)
        )
        sales.loc[:, f"rolling_sales_std_{window}"] = (
            sales.groupby(["id"])["sales"]
            .transform(lambda x: x.rolling(window=window).std())
            .astype(np.float16)
            .fillna(0)
        )

    # difference between short-long trends
    short_windows = [3, 7, 30]
    long_windows = [90, 180, 360]

    for short_window in tqdm(short_windows, desc="fe: long-short term diff"):
        for long_window in long_windows:
            sales.loc[:, f"trend_diff_{short_window}_{long_window}"] = (
                sales[f"rolling_sales_mean_{short_window}"]
                - sales[f"rolling_sales_mean_{long_window}"]
            )

    # store-wise rolling average

    # see if in a zero plateau
    sustained_zero = (
        sales.groupby("id")["sales"]
        .apply(lambda x: intervaled_cumsum(x == 0))
        .explode()
        .astype(int)
    )
    sustained_zero.name = "days_sustained_zero"
    sustained_zero = sustained_zero.reset_index()
    sustained_zero.loc[:, "day"] = sustained_zero.groupby("id").transform(
        lambda x: np.arange(len(x)) + 1
    )

    sales = sales.merge(sustained_zero, on=["id", "day"], how="left", copy=False)

    # create target variable
    for ahead in tqdm(ahead_windows, desc="fe: ahead target"):
        y_ahead = create_shifted_feature(sales, ahead)
        sales.loc[:, f"target_shift_{ahead}"] = y_ahead

    # shift future events back
    for ahead in tqdm(ahead_windows, desc="fe: ahead events"):
        event_type_ahead = create_shifted_feature(
            sales, ahead, feature_col="event_type"
        ).replace({0: "No events"})
        event_name_ahead = create_shifted_feature(
            sales, ahead, feature_col="event_name"
        ).replace({0: "No events"})
        weekday_ahead = create_shifted_feature(sales, ahead, feature_col="weekday")

        weekend_ahead = weekday_ahead.isin({0, 1}).astype(int)
        friday_ahead = (weekday_ahead == 6).astype(int)

        sales.loc[:, f"event_type_shift_{ahead}"] = event_type_ahead
        sales.loc[:, f"event_name_shift_{ahead}"] = event_name_ahead
        sales.loc[:, f"weekend_shift_{ahead}"] = weekend_ahead
        sales.loc[:, f"friday_shift_{ahead}"] = friday_ahead

    return sales

In [ ]:
AHEAD_WINDOWS = [3, 7, 14, 28]

In [ ]:
data_saved = os.path.isfile("../data/train.h5")
# data_saved = False
if data_saved:
    data = pd.read_hdf("../data/train.h5")
else:
    data = merge_and_melt(sales, calendar, prices)
    data = feature_engineering(data, AHEAD_WINDOWS)
    data = reduce_mem_usage(data)

    data.to_hdf("../data/train.h5", key="train", format="table")

In [ ]:
dump_cols = [
    "id",
    "wm_y_wk",
    "event_name",
    "d",
    "day",
    "event_type",
    "wm_yr_wk",
    "weekday",
    "date",
] + [c for c in data.columns if c.startswith("target")]
feat_cols = [c for c in data.columns if c not in dump_cols]
cat_cols = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
    "event_name_shift_3",
    "event_name_shift_7",
    "event_name_shift_14",
    "event_name_shift_28",
    "event_type_shift_3",
    "event_type_shift_7",
    "event_type_shift_14",
    "event_type_shift_28",
    "weekend_shift_3",
    "friday_shift_3",
    "weekend_shift_7",
    "friday_shift_7",
    "weekend_shift_14",
    "friday_shift_14",
    "weekend_shift_28",
    "friday_shift_28",
]

cont_cols = [c for c in feat_cols if c not in cat_cols]

In [ ]:
### ENCODE CATEGORICAL VARIABLES
encode_dict = {}

for col in cat_cols:
    lbe = LabelEncoder()
    lbe.fit(data[col])
    data.loc[:, col] = lbe.fit_transform(data[col])
    encode_dict[col] = lbe

pickle.dump(encode_dict, open("../models/encoders.pkl", "wb"))

In [ ]:
cat_mean_dict = {col: train.groupby(col)["sales"].mean().to_dict() for col in cat_cols}

In [ ]:
cat_mean_dict['']

### Bayseian Ridge

In [ ]:
from sklearn.linear_model import BayesianRidge
from sklearn.preprocessing import StandardScaler

In [ ]:
val_preds = []
test_preds = []

bayes_models = []

for ahead_by in tqdm(AHEAD_WINDOWS, leave=False):
    target_col = f"target_shift_{ahead_by}"

    train = data[data["day"] < 1914 - ahead_by]  # need to subtract lag used
    val = data[(1914 - ahead_by <= data["day"]) & (data["day"] < 1914)]
    test = data[(data["day"] >= 1914) & (data["day"] < 1942)]

    for col, mean in cat_mean_dict.items():
        train[col] = train[col].replace(mean)
        val[col] = val[col].replace(mean)
        test[col] = test[col].replace(mean)

    X_train, y_train = train[feat_cols], train[target_col]
    X_val, y_val = val[feat_cols], val[target_col]
    X_test, y_test = test[feat_cols], test[target_col]

    ### TRAIN LGBM MODEL
    preproc = StandardScaler()
    X_train.loc[:, cont_cols] = preproc.fit_transform(X_train[cont_cols])
    X_val.loc[:, cont_cols] = preproc.transform(X_val[cont_cols])
    X_test.loc[:, cont_cols] = preproc.transform(X_test[cont_cols])

    bayes_model = BayesianRidge()
    bayes_model.fit(X_train, y_train)

    pickle.dump(bayes_model, open(f"../models/bayes_{ahead_by}.pkl", "wb"))

    bayes_models.append(bayes_model)

    val_pred = bayes_model.predict(X_val)
    test_pred = bayes_model.predict(X_test)

    val_preds.append(val_pred)
    test_preds.append(test_pred)
#     break

In [ ]:
shortest_length = len(min(val_preds, key=len))
val_pred_bayes = np.mean(
    [val_pred[-shortest_length:] for val_pred in val_preds], axis=0
)
test_pred_bayes = np.mean(test_preds, axis=0)

In [ ]:
mean_squared_error(y_val.values[-shortest_length:], val_pred_bayes, squared=False)

### LGBM

In [ ]:
model_params = {
    "n_estimators": 1000,
    "num_leaves": 100,
    "max_depth": 9,
    "learning_rate": 0.03,
    "boosting_type": "gbdt",
    "objective": "poisson",
    "metric": "rmse",
    "reg_alpha": 0.2,
    "reg_lambda": 0.2,
    "subsample": 0.6,
    "categorical_feature": [
        i for i in range(len(feat_cols)) if feat_cols[i] in cat_cols
    ],
    "min_split_gain": 0.02,
}

In [ ]:
val_preds_lgbm = []
test_preds_lgbm = []

lgbm_models = []

for ahead_by in tqdm(AHEAD_WINDOWS, leave=False):
    target_col = f"target_shift_{ahead_by}"

    train = data[data["day"] < 1914 - ahead_by]  # need to subtract lag used
    val = data[(1914 - ahead_by <= data["day"]) & (data["day"] < 1914)]
    test = data[(data["day"] >= 1914) & (data["day"] < 1942)]

    X_train, y_train = train[feat_cols], train[target_col]
    X_val, y_val = val[feat_cols], val[target_col]
    X_test, y_test = test[feat_cols], test[target_col]

    ### TRAIN LGBM MODEL
    train_dataset = lgbm.Dataset(X_train, y_train)
    val_dataset = lgbm.Dataset(X_val, y_val)

    lgbm_model = lgbm.train(
        model_params,
        train_dataset,
        300,
        valid_sets=[train_dataset, val_dataset],
        verbose_eval=100,
        early_stopping_rounds=100,
    )

    pickle.dump(lgbm_model, open(f"../models/lgbm_poisson_{ahead_by}.pkl", "wb"))

    lgbm_models.append(lgbm_model)

    val_pred = lgbm_model.predict(X_val)
    test_pred = lgbm_model.predict(X_test)

    val_preds_lgbm.append(val_pred)
    test_preds_lgbm.append(test_pred)
#     break

In [ ]:
shortest_length = len(min(val_preds, key=len))
val_pred_lgbm = np.mean(
    [val_preds_lgbm[1][-shortest_length:], val_preds_lgbm[3][-shortest_length:]], axis=0
)
test_pred_lgbm = np.mean([test_preds_lgbm[1], test_preds_lgbm[3]], axis=0)

In [ ]:
mean_squared_error(
    y_val.values[-shortest_length:], val_pred_lgbm, squared=False,
)

### format to submission

In [ ]:
qs = np.array([0.005, 0.025, 0.165, 0.25, 0.5, 0.75, 0.835, 0.975, 0.995])


def get_ratios(coef=0.15):
    qs2 = np.log(qs / (1 - qs)) * coef
    ratios = stats.norm.cdf(qs2)
    ratios /= ratios[4]
    ratios = pd.Series(ratios, index=qs)
    return ratios.round(3)


def merge_pred_with_submission(pred, test_df):

    test_df.loc[test_df.index, "pred"] = pred
    submission_test = test.groupby("id")["pred"].apply(list).apply(pd.Series)
    submission_test.columns = ["F" + str(i + 1) for i in submission_test.columns]
    submission_test = submission_test.reset_index()
    return submission_test


# coef between 0.03 and 0.25 is used, probably suboptimal values for now
# If you're not sure about your Accuracy submission, you should make the distribution wider.
level_coef_dict = {
    "_all_": get_ratios(coef=0.03),
    "state_id": get_ratios(coef=0.04),
    "store_id": get_ratios(coef=0.05),
    "cat_id": get_ratios(coef=0.04),
    "dept_id": get_ratios(coef=0.05),
    ("state_id", "cat_id"): get_ratios(coef=0.05),
    ("state_id", "dept_id"): get_ratios(coef=0.07),
    ("store_id", "cat_id"): get_ratios(coef=0.07),
    ("store_id", "dept_id"): get_ratios(coef=0.08),
    "item_id": get_ratios(coef=0.11),
    ("state_id", "item_id"): get_ratios(coef=0.15),
    "id": get_ratios(coef=0.25),
}

levels = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id", "_all_"]
couples = [
    ("state_id", "item_id"),
    ("state_id", "dept_id"),
    ("store_id", "dept_id"),
    ("state_id", "cat_id"),
    ("store_id", "cat_id"),
]
cols = [f"F{i}" for i in range(1, 29)]


def quantile_coefs(q, level):
    ratios = level_coef_dict[level]
    return ratios.loc[q].values


def get_group_preds(pred, level):
    df = pred.groupby(level)[cols].sum()
    q = np.repeat(qs, len(df))
    df = pd.concat([df] * 9, axis=0, sort=False)
    df.reset_index(inplace=True)
    df[cols] *= quantile_coefs(q, level)[:, None]
    if level != "id":
        df["id"] = [
            f"{lev}_X_{q:.3f}_evaluation" for lev, q in zip(df[level].values, q)
        ]
    else:
        df["id"] = [
            f"{lev.replace('_evaluation', '')}_{q:.3f}_evaluation"
            for lev, q in zip(df[level].values, q)
        ]
    df = df[["id"] + list(cols)]
    return df


def get_couple_group_preds(pred, level1, level2):
    df = pred.groupby([level1, level2])[cols].sum()
    q = np.repeat(qs, len(df))
    df = pd.concat([df] * 9, axis=0, sort=False)
    df.reset_index(inplace=True)
    df[cols] *= quantile_coefs(q, (level1, level2))[:, None]
    df["id"] = [
        f"{lev1}_{lev2}_{q:.3f}_evaluation"
        for lev1, lev2, q in zip(df[level1].values, df[level2].values, q)
    ]
    df = df[["id"] + list(cols)]
    return df


def get_submission_format(sub):

    levels = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id", "_all_"]
    couples = [
        ("state_id", "item_id"),
        ("state_id", "dept_id"),
        ("store_id", "dept_id"),
        ("state_id", "cat_id"),
        ("store_id", "cat_id"),
    ]
    cols = [f"F{i}" for i in range(1, 29)]

    df = []
    for level in levels:
        df.append(get_group_preds(sub, level))
    for level1, level2 in couples:
        df.append(get_couple_group_preds(sub, level1, level2))
    df = pd.concat(df, axis=0, sort=False)
    df.reset_index(drop=True, inplace=True)

    return df


submission_test = merge_pred_with_submission(test_pred_lgbm, test)

sub_test = submission_test.merge(
    sales[["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]], on="id"
)
sub_test["_all_"] = "Total"

####
# need to append validation
####
# df_val = get_submission_format(sub_val)
df_eval = get_submission_format(sub_test)
df_val = df_eval

df = pd.concat([df_eval, df_val], axis=0, sort=False)
df.reset_index(drop=True, inplace=True)
df.loc[df.index >= len(df.index) // 2, "id"] = df.loc[
    df.index >= len(df.index) // 2, "id"
].str.replace("_evaluation$", "_validation")

df.to_csv("submission_lgbm_7_28.csv", index=False)

In [ ]:
plot_importance(lgbm_models[-1], figsize=(10, 10))